# Observational Descriptive Research

**Topic 04 · 2 lectures**

<hr>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb04_observational_descriptive_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** description → generalization (descriptive kind · data-at-hand
and population reach). A **descriptive question** asks what is actually there in
the cases you can see. Example: "in the buildings my sensors cover, what share
exceed the air-quality threshold?" This week you also test when a description of
your sample is allowed to become a statement about a wider **population**, the
larger group your question is really about.

**Design pathway:** observational descriptive. **Observational** means you watch
and record without changing what anyone experiences. You never assign a
treatment, so this pathway can describe and, with care, generalize, but it
cannot by itself say one thing caused another.

| | |
|---|---|
| **A claim this topic PERMITS** | "In the units my sampling frame actually reaches, the share with characteristic X is [estimate], described exactly as observed; and where sampling and response allow, the population estimate is [value] with stated uncertainty and named coverage limits." |
| **A claim this topic does NOT permit** | "This describes the whole population" when the frame misses part of it, "the average among my respondents is the population average" when nonresponse relates to what is measured, or "X causes Y" from a design that never intervened. |

**Where this sits in the course:** Week 5 of the design library. It builds on the
research question you declared in M3 and develops M4, your observational
descriptive design audit, which you present at this week's Friday studio. M4
then hands off to M5, where the causal question opens.

*Provenance: RDSS ch. 15 'Observational: descriptive' + the DeclareDesign observational-descriptive design library + rdss la_voter_file and lapop_brazil | population/frame/sample stack, measurement and nonresponse, the generalization boundary | seeded frame→sample draws and a seeded nonresponse stress-test (seed 464) | adapted (R→Python, honors non-quant audience)*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish the **target population**, the **accessible population**, the
   **sampling frame**, and the **sample**, and say which one any claim silently
   invokes.
2. Draw a **selection DAG** that shows where the people you meant to study
   drop out of the data you actually observe.
3. Show, by seeded simulation, that random draws from a frame cluster around its
   truth while a convenience channel can miss it badly, and explain why.
4. Summarize a real survey honestly with a distribution, a relationship, and an
   index, and read what each reveals that a single average hides.
5. Diagnose how **nonresponse** bends a description, and state the exact boundary
   past which your sample may not speak for a population.
6. Apply the population/frame/sample stack and a coverage-and-nonresponse note to
   your own project's M4 observational descriptive design audit.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx  # for the selection DAGs; if missing locally: pip install networkx (pre-installed in Colab)

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)  # used for the jitter on the ten-means strip below

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

### 🤝 AI Research Partner

This week you work with **Gemini** as a research partner, not an answer machine.
A good use looks like this: you commit your own answer first, then ask Gemini to
explain a result, name a gap you might have missed, or draft a caveat you will
check line by line. A bad use is letting it decide your population, wave a claim
past your evidence, or hand you a source you never opened.

**Never delegate these:** which population your project is about, which frame you
can honestly reach, and where your description must stop. Those are yours to
declare and defend. The full list of never-delegate decisions lives in
[`ai_resources/human_responsibility_checklist.md`](https://github.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/ai_resources/human_responsibility_checklist.md).

**Commit first, then ask.** Every AI block below asks you to write your own
answer before you open Gemini. And commit your work to git before a long AI
session, so you can always compare what you wrote against what the tool changed.

> **A question that often comes up here:** *"If Gemini is faster and often
> right, why should I write my answer first?"* Because the habit protects your
> judgment. If you read the tool's answer first, it anchors you, and you lose
> the one thing you bring that it cannot: knowing what your project is really
> about. Writing first keeps you the researcher and keeps Gemini the assistant.

---

# Lecture 1

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

A campus poll stops people walking out of one dining hall at lunch and reports
the result as **"what everyone at Purdue thinks."** The poll is real, the math is
correct, and hundreds of people answered.

Here is the claim on the table: **"This poll speaks for everyone at Purdue."**
Would you stake your name on it, yes or no? Commit to one answer before we go
further. Then defend it: name three specific groups the poll never had a chance
to reach, and for each, say whether they would plausibly answer differently on
the very thing being measured. If even one such group exists, your yes is in
trouble. Naming those groups precisely is the whole job of Lecture 1.

## 1. Who Gets Into Your Data: Population, Frame, Sample

**Guiding question:** *which group of people does your evidence actually speak
for: the one you wish you studied, or the one your procedure really reached?*

> *"Every result you bring me answers a question about some group. My first
> question is always the same. Which group? Not the group you hoped to study.
> The one your procedure actually reached."*
> — a **policy stakeholder** deciding whether your evidence applies to their city

Your **data strategy** is your plan for how evidence comes to exist: who ends up
in your data, and what you record about them. This week the pathway is
**observational**, meaning you watch and record without changing anything. Of all
the steps in a data strategy, only two decide who you actually see, and those two
are the whole job this week.

- **Sampling (S)**: how units are drawn into your study. It decides who even gets
  a chance to appear in your data. Example: which 100 voters you contact from a
  list of thousands.
- **Response (R)**: who, once drawn, actually gives you data. It decides who among
  the people you reached ends up recorded. Example: of the 100 you contacted, the
  62 who answered.

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_8_1.png" width="70%"/></center>

*The figure lays out the full data strategy, but this week you touch only two of
its handles: **sampling (S)**, who gets drawn in, and **response (R)**, who then
answers. Observational descriptive research lives on those two. You sample and you
record, and you never assign anyone a treatment. (Figure from the replication
materials of Blair, Coppock & Humphreys (2023),* Research Design in the Social
Sciences*, an MIT-licensed archive; the book is free at book.declaredesign.org.)*

Four words keep an observational description honest. Picture four boxes, each
smaller than the last:

- **Target population**: everyone your question is *about*. Example: all
  registered voters in Los Angeles. This is the outermost box, the group you
  wish you could study.
- **Accessible population**: the part of the target you could reach in
  principle, given time, cost, and access. Example: LA voters who still have a
  valid address on file. Newly moved voters slip out of this box.
- **Sampling frame**: the concrete list you actually draw names from. Example: a
  voter file. Whoever is not on the list cannot be sampled, no matter how good
  your procedure is.
- **Sample**: the units you end up observing. The innermost box.

A claim is honest only when the box it describes matches the box it names. You
will work with a real **frame** today: an extract of the Los Angeles voter file.
For this notebook, treat the file itself as your known world, so you can *see*
the truth and watch sampling try to recover it.

> **A question that often comes up here:** *"My list is huge. Isn't the frame
> basically the population at that point?"* Almost never, and size is what hides
> the gap. A voter file can hold millions of names and still miss the people who
> just moved, just turned eighteen, or were purged last month. If those missing
> people differ on what you measure, no amount of list length closes the gap.
> The frame is the population minus everyone the list forgot.

Three more words name *why* the boxes differ, and all three run through the rest
of the course.

- **Selection**: any process that decides who lands in your data instead of
  chance alone. Example: a poll taken outside one dining hall selects for people
  who eat there at lunch. Selection is the arrow that pulls certain people in and
  leaves others out.
- **Coverage**: how well your frame fills the population you care about. Example:
  a voter file has good coverage of long-term residents and poor coverage of
  new arrivals. A coverage gap is the slice of the population your frame never
  had a chance to include.
- **Silent exclusion**: a group your frame drops without ever announcing it.
  Example: a survey sent through a dorm email list silently excludes everyone who
  opted out of dorm email. No one chose to leave them out. The frame simply never
  listed them. Silent exclusions are dangerous because they stay invisible until
  you name them out loud, and naming yours is a graded step in your M4 audit.

Load the frame and read its truth. Because you can see the whole file, you know
the answer sampling is trying to find.

In [ ]:
# Load the voter file and treat it as our known frame.
voters = load_course_data("la_voter_file.csv")
assert voters.shape == (1000, 4), "unexpected shape — flag this!"
print("✓ Loaded la_voter_file.csv —", voters.shape[0], "rows,", voters.shape[1], "columns")
print("  Columns:", list(voters.columns))
print()

true_mean_age = voters["age"].mean()
print("The whole frame (our 'known world'):")
print(f"  mean age     = {true_mean_age:.2f}   (min {voters['age'].min()}, max {voters['age'].max()})")
print(f"  turnout 2012 = {voters['voted_2012'].mean():.1%} voted")
print(f"  party mix    = {voters['party'].value_counts().head(3).to_dict()}")

### 🔍 Reading the Evidence: name the boxes

In the cell below, answer three things. First: for the LA voter file, which box
is the file itself, the target population, the frame, or a sample? Defend the
label. Second: name one kind of person who is in the *target population*
"registered LA voters" but could be missing from this frame. Third: the frame's
mean age is about 48.8. Is that the target population's mean age? Name the one
word that separates "the frame's average" from "the population's average," and
say why you cannot skip it.

### YOUR ANSWER HERE:

**Which box is the voter file, and why:**

**Someone in the target population but missing from the frame:**

**Is 48.8 the population's mean age? The word I cannot skip:**

---

## 2. The Selection DAG: Where People Drop Out

**Guiding question:** *at which step do the people you meant to study quietly
leave the data you actually observe?*

A **selection diagram**, also called a **DAG** (short for directed acyclic graph,
a picture of arrows where each arrow means "this influences that"), shows how
someone travels from the target population into your observed sample, and where
they can fall out along the way. This course calls it a **selection DAG** from
here on, the same name your M4 audit uses. Each arrow is a chance to lose people
who differ from the ones who stay. Drawing it before you collect data is how you
find a coverage problem while you can still fix it.

**🏠 Homework depth — run this on your own before the next lecture.**
**Before you ask:** in one sentence, name the single group you most fear your own
study will miss. Write it down first, so you can see whether the tool finds a
group you did not.

> 💡 **Gemini Prompt:** "I am building a selection DAG for an observational study.
> My target population is [your target], the accessible population is [who I can
> reach], my sampling frame is [my list], and my sample is [who I observe]. List
> the specific groups that drop out at each arrow, in a table with one column for
> the group and one for whether their absence pushes my estimate up or down. Then,
> as a hostile reviewer, tell me the one group on your list I am most likely wrong
> to include, and the one important group you suspect you left out."
>
> **After running, verify (counters *illusion of completeness*: a long, tidy list
> that quietly omits the group that matters most):**
> - [ ] Compare the list against the group you committed to above. If yours is
>   missing, the tidy output just failed the one test it looked complete on.
> - [ ] Map each group it names to one arrow in the selection DAG you draw below.
>   A group that fits no arrow is one the tool invented for you.
> - [ ] Decide *yourself* whether each named group plausibly differs on your
>   outcome. The tool lists groups; only you know your outcome's mechanism.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Draw the selection DAG: target -> accessible -> frame -> observed.
# (networkx is imported once in the setup cell.)
G = nx.DiGraph()
pos = {
    "Target\npopulation":     (0.0, 1.0),
    "Accessible\npopulation": (1.5, 1.0),
    "Sampling\nframe":        (3.0, 1.0),
    "Observed\nsample":       (4.5, 1.0),
    "Selection\ninfluence":   (3.0, 2.2),
}
edges = [
    ("Target\npopulation", "Accessible\npopulation"),
    ("Accessible\npopulation", "Sampling\nframe"),
    ("Sampling\nframe", "Observed\nsample"),
    ("Selection\ninfluence", "Observed\nsample"),
]
G.add_edges_from(edges)

fig, ax = plt.subplots(figsize=(10, 4.5))
nx.draw_networkx_nodes(G, pos, node_color="#DCE6F1", edgecolors="#4C72B0",
                       linewidths=1.5, node_size=5200, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
nx.draw_networkx_edges(G, pos, arrowstyle="-|>", arrowsize=18, edge_color="#555555",
                       width=1.6, node_size=5200, ax=ax)
edge_labels = {
    ("Target\npopulation", "Accessible\npopulation"): "access/reach gap",
    ("Accessible\npopulation", "Sampling\nframe"): "coverage / frame error",
    ("Sampling\nframe", "Observed\nsample"): "sampling / selection",
}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8,
                             label_pos=0.5, ax=ax)
ax.set_title("A selection DAG: where the people you meant to study drop out")
ax.axis("off")
plt.tight_layout()
plt.show()

print("✓ Selection DAG drawn.")
print("  Each arrow is a place people can drop out before you ever observe them.")
print("  The top arrow is the dangerous one: a selection influence bends both who")
print("  you reach and what you measure, so your sample tilts on the very outcome.")

**Reading the diagram.** Follow one person from the leftmost box to the
rightmost. They start in the target population. They survive to the accessible
population only if you can reach them at all. They enter the frame only if the
list includes them. They enter your sample only if selection draws them. The top
arrow carries the warning. A **selection influence** is a trait that makes someone
both easy to include *and* different on what you measure. When one trait does
both, your sample tilts toward one kind of person on the very outcome you care
about. The next section makes that tilt visible.

*In plain terms, the picture says every step from left to right is a filter, and
the top arrow is the filter that can quietly bias your answer.*

## 3. Random Draws Cluster; Convenience Channels Select

**Guiding question:** *why can you trust a random draw from a frame to stand in
for it, while a convenient channel can mislead you no matter how large it grows?*

You know this frame's true mean age, 48.8. Now watch two ways of sampling try to
recover it. **Random sampling** means chance, not convenience, decides who enters
your data: every unit on the frame has a known chance of being drawn. Example:
numbering all 1,000 rows and letting the computer pick 100 at random.

> **A question that often comes up here:** *"My convenience sample is big.
> Doesn't a big sample fix the bias?"* No, and this is the trap. Size shrinks
> *noise*, the sample-to-sample wobble, but it does nothing to *bias*, a
> systematic tilt in who you reach. A huge convenience sample gives you a very
> precise estimate of the wrong number. You will see exactly that below.

### 🔮 Pause & Predict

You are about to draw **10 random samples of 100 voters each** and compute each
sample's mean age. The frame's true mean is 48.8. Before running the next cell,
commit to a range. Between what low and what high value do you think the ten
sample means will fall? Will they scatter widely, or huddle near 48.8? Write one
sentence of reasoning.

### YOUR ANSWER HERE:

**My predicted range for the 10 sample means (low to high):**

**Scattered or huddled, and why:**

---

### 🛠️ Run the Study: draw ten random samples

Run the cell below. It draws ten independent random samples of 100 voters from
the frame, computes each sample's mean age, and plots the ten means against the
frame's true mean. Read the spread before you read the next markdown cell. If you
want, change `n=100` to `n=500` and rerun to feel the spread tighten.

**🔴 Live in class: we run this one together.**
**Before you ask:** write one sentence predicting whether the ten means will
huddle near 48.8 or scatter widely (you set a range last cell; now commit the shape).

> 💡 **Gemini Prompt:** "Here is a Python cell that draws ten random samples of
> 100 voters and plots each sample's mean age against the true mean: [paste the
> next cell]. Explain, line by line, what the loop computes and why the ten dots
> land in a band around the true line instead of on top of it. Then give me one
> independent way to confirm the ten means really center on the frame's true mean,
> without rerunning your own code."
>
> **After running, verify (counters *confident fabrication*: a fluent line-by-line
> walkthrough can describe a result the code never produced):**
> - [ ] Confirm the lowest-to-highest range Gemini quotes matches the `lowest` and
>   `highest` values your cell actually printed, not a number it guessed.
> - [ ] Run its "independent way to confirm" yourself. If it will not run, or gives
>   a different center than your printout, trust your printout.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Ten random samples of n=100, each with its own reproducible seed.
sample_means = np.array([
    voters["age"].sample(n=100, random_state=SEED + i).mean()
    for i in range(10)
])
print("Ten random-sample mean ages (n = 100 each):")
print(np.round(sample_means, 2).tolist())
print(f"\n  lowest  = {sample_means.min():.2f}")
print(f"  highest = {sample_means.max():.2f}")
print(f"  true frame mean = {true_mean_age:.2f}")

# Small vertical jitter (drawn from the course rng) so overlapping means separate.
jitter = rng.uniform(-0.12, 0.12, size=sample_means.size)

fig, ax = plt.subplots()
ax.scatter(sample_means, jitter, s=90, color="#4C72B0",
           zorder=3, label="each random sample's mean")
ax.axvline(true_mean_age, color="#C44E52", linestyle="--", linewidth=2,
           label=f"true frame mean = {true_mean_age:.1f}")
ax.set_yticks([])
ax.set_ylim(-0.5, 0.5)
ax.set_xlim(30, 58)
ax.set_xlabel("Sample mean age")
ax.set_title("Random samples huddle around the frame's truth (n = 100, 10 draws)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: every random-sample mean lands close to the truth.
assert sample_means.min() > 42 and sample_means.max() < 55, "spread wider than expected"
assert abs(sample_means.mean() - true_mean_age) < 3, "random samples should center on the truth"
print(f"✓ Self-check passed: all 10 random-sample means fall in "
      f"[{sample_means.min():.1f}, {sample_means.max():.1f}], a tight band around {true_mean_age:.1f}.")

The ten means all land in a tight band around 48.8. They *huddle*. The self-check
below prints the exact lowest and highest, so you can read off how narrow that
band really is rather than trust a vague "close." Because random draws cluster
this tightly, a single random sample of 100 licenses a careful claim: the frame's
mean age is near my sample mean, give or take a stated margin. The missing
ingredient that turns "my sample mean" into "a statement about the frame" is
**uncertainty**, an interval or margin of error, which the inference week
formalizes. Random sampling is what earns the right to attach one.

Now the other way. A **convenience channel** reaches whoever was easiest to find:
one dining hall, one club, one mailing list. Below, imagine your survey went out
through one convenient outreach list, and the only people it reached were
registrants with no party preference (the NPP group). You did not choose them for
their age. The list simply reached them. Watch what it does to the age you would
report anyway.

### 🔮 Pause & Predict

The next cell keeps only the NPP registrants and reports their mean age next to
the frame's true 48.8, on the same axis as your ten random samples. Before you run
it, commit: will the convenience mean land *inside* the random band, or outside
it? If outside, which way, older or younger? Write one sentence of reasoning, and
do not peek at the output first. You are predicting a reveal.

### YOUR ANSWER HERE:

**Inside the random band or outside it, and which direction:**

**One sentence of reasoning:**

---

**🏠 Homework depth: run this on your own after class.**
**Before you ask:** in one sentence, name the mechanism you think links this
convenience group to age. Commit your own guess before the tool offers one.

> 💡 **Gemini Prompt:** "Here is a cell that builds a convenience sample by keeping
> only one group of registrants from a voter file, then compares its mean age to
> the whole file's: [paste the next cell]. Walk me through, step by step, how a
> sample can be large and precise and still systematically wrong. Then, without me
> telling you the answer, propose what trait of this group could be pulling its
> mean age away from the frame, and how I could check that trait from the data."
>
> **After running, verify (counters *silent scope change*: 'the sample is biased'
> is not the same claim as 'this specific trait biased it'):**
> - [ ] Check Gemini's numbers against your printout. Does the mean it discusses
>   match the convenience mean your cell reported, and the size of the gap in years?
> - [ ] Do not accept a bare "it is biased." Confirm its proposed mechanism against
>   the by-party table your cell prints, and reject it if the table does not support it.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# A convenience channel reaches one kind of person, not a random draw.
# First, just read mean age by party in this frame. Draw no conclusion yet.
print("Mean age by party in this frame:")
print(voters.groupby("party")["age"].mean().round(1)
      .loc[["NPP", "DEM", "REP"]].to_string())
print()

# The convenience list reached only the no-party-preference (NPP) registrants.
convenience = voters[voters["party"] == "NPP"]["age"]
conv_mean = convenience.mean()
print(f"Convenience channel (all {len(convenience)} NPP registrants):")
print(f"  mean age = {conv_mean:.2f}   vs frame truth {true_mean_age:.2f} "
      f"-> off by {conv_mean - true_mean_age:+.1f} years")

# Put the convenience mean next to the random-sample band from before.
# The convenience marker gets its OWN hue so it never blends into the truth line.
fig, ax = plt.subplots()
ax.scatter(sample_means, jitter, s=90, color="#4C72B0",
           zorder=3, label="random samples (n = 100 each)")
ax.scatter([conv_mean], [0], s=200, color="#DD8452", marker="X", zorder=4,
           label=f"convenience channel = {conv_mean:.1f}")
ax.axvline(true_mean_age, color="#C44E52", linestyle="--", linewidth=2,
           label=f"true frame mean = {true_mean_age:.1f}")
ax.set_yticks([])
ax.set_ylim(-0.5, 0.5)
ax.set_xlim(30, 58)
ax.set_xlabel("Mean age")
ax.set_title("Bias, not noise: the convenience channel sits outside the whole random band")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the convenience mean sits OUTSIDE the entire random-sample band.
assert abs(conv_mean - 37.0) < 0.5, "NPP mean drifted — investigate"
assert conv_mean < sample_means.min(), "convenience mean should fall below every random-sample mean"
print(f"✓ Self-check passed: the convenience mean ({conv_mean:.1f}) is below the lowest "
      f"random-sample mean ({sample_means.min():.1f}). This gap is bias, not noise.")

### 🔍 Reading the Evidence: what the convenience channel would have you believe

The convenience channel would tell you these voters average about 37, nearly 12
years younger than the truth. No amount of extra registrants would fix it. In
the cell below, write three things: the false claim this channel invites, the
*mechanism* that biased it (why did reaching one party skew age?), and one honest
sentence the convenience sample CAN support without lying.

### YOUR ANSWER HERE:

**The false claim it invites:**

**The mechanism behind the bias:**

**One honest sentence it can still support:**

---

---

### ⏸ In class through here

**You have reached the end of Lecture 1's in-class block.** Everything below
deepens the same ideas. Finish it on your own before the next lecture: the design
choice, the four-study practice drill, and the two homework-depth Gemini prompts
you met above. Come back with your M4 population/frame/sample stack started.

---

### ⚖️ Make a Design Choice: which frame do you sample?

*(Human-first: settle your own choice and its defense before you ask any AI.)*

You want to describe the air quality inside Purdue's academic buildings, for the
question *what share of academic buildings exceed the healthy CO₂ threshold
during class hours?* You have three possible frames. **Choose one and defend it**
in a short paragraph, naming who each frame excludes and whether that exclusion
relates to the outcome:

- **A.** Only the buildings that already have permanent air sensors installed.
- **B.** The registrar's full list of academic buildings, from which you draw a
  random sample to visit with a portable sensor.
- **C.** The buildings you personally have classes in this semester.

In the cell below, pick A, B, or C, then defend the choice: what does your chosen
frame let you claim, and where must that claim stop?

### YOUR ANSWER HERE:

**My chosen frame (A / B / C):**

**Who it excludes, and whether that exclusion relates to CO₂ levels:**

**What my description can claim, and where it must stop:**

---

### 📝 Practice: target, accessible, selection for four observational studies

*(Human-first: classify all four yourself before any AI. You may check with AI
after, but the sorting is the skill being graded.)*

Observational description is everywhere in STEM, not just surveys. For each
setting below, name the **target population**, the **accessible population**, and
the **selection** that separates them. Answer in the cell that follows.

- **A. Environmental monitoring.** A team reports river-water nitrate levels for
  a watershed, using the sampling stations that already exist along the banks.
- **B. Biodiversity survey.** A field crew records bird species by walking marked
  trails through a forest at dawn and logging what they hear.
- **C. Materials characterization.** A lab reports the defect rate of a new alloy
  from the test coupons that survived cutting and polishing intact.
- **D. Usage telemetry.** An app team describes "how users behave" from the event
  logs of people who left analytics tracking switched on.

### YOUR ANSWER HERE:

**A (target / accessible / selection):**

**B (target / accessible / selection):**

**C (target / accessible / selection):**

**D (target / accessible / selection):**

---

You have built the population/frame/sample stack and drawn where selection
enters. Lecture 2 turns to the data once they are in hand: how to summarize them
honestly, how nonresponse bends the summary, and exactly where a description of
your sample is allowed to become a claim about a population.

---

---

# Lecture 2

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

A national survey is fielded, and **40% of the people contacted never answer.**
The report still prints one confident line: "62% of the country approves,
margin of error 2 points."

The margin of error looks reassuring. Before we go further, name what it does
*not* cover. If the 40% who stayed silent differ from the 60% who answered on the
very thing being asked, what happens to that 62%? What single piece of evidence
would you need before you would trust the number as a *national* estimate rather
than a description of the people who happened to answer? Write your answer, then
hold it. Lecture 2 is about restoring, or refusing, that missing license.

## 4. Reading the Sample: Distributions, Relationships, Indices

**Guiding question:** *once the data are in, how do you summarize them honestly,
before you ask how far the summary may reach?*

> *"Show me your Table 1 before you show me your finding. If you cannot describe
> your sample honestly, I will not believe anything you built on top of it."*
> — a **journal reviewer**, on the first draft they will ever read

Description is where a reader's trust is won or lost. Three honest summaries do
most of the work, and each answers a different question:

- **Distribution**: the full lineup of answers to one variable, how many people
  gave each value. Example: how many rated their trust in police a 1, a 2, and so
  on up to 7. A distribution shows the shape an average hides.
- **Relationship**: how two variables move together. Example: do people who trust
  the police more also tend to trust the military more? A relationship is a
  pattern, not yet a cause.
- **Index**: several related items combined into one measure. Example: averaging
  three institutional-trust items into a single "trust in institutions" score. An
  index is only as honest as the items inside it.

Behind all three sits **measurement**: turning something abstract into a recorded
number. Measurement climbs a three-rung ladder, and each rung narrows the one
above it.

- **Concept**: the broad idea you care about. Example: "trust in institutions."
- **Construct**: the specific, targetable version of that idea. Example: "trust in
  the national police."
- **Indicator**: the concrete thing you can actually record. Example: a 1-to-7
  survey item asking how much you trust the police.

A concept you cannot see is only ever reached through an indicator you can, and
the indicator is never the concept itself. Your M4 audit climbs this same
**concept → construct → indicator** ladder for your own key construct, so name the
rungs as you read the survey below.

In [ ]:
# Load a real survey and confirm its shape before trusting anything.
lapop = load_course_data("lapop_brazil.csv")
assert lapop.shape == (10000, 10), "unexpected shape — flag this!"
print("✓ Loaded lapop_brazil.csv —", lapop.shape[0], "rows,", lapop.shape[1], "columns")
print("  Trust items are on a 1 (none) to 7 (a lot) scale.")
print("  Columns:", [c for c in lapop.columns if c != "ID"])

This file is real AmericasBarometer (LAPOP) survey data from Brazil, and it
carries one catch you must state every time you quote a number from it.

> **A question that often comes up here:** *"If this is real survey data, why
> can't I just say what Brazilians think?"* Because this file is a
> **10,000-row resample with replacement** of the original interviews. In plain
> terms, the package built it by drawing rows from the real survey over and over,
> like pulling names from a hat and dropping each name back after every draw. That
> makes it perfect for learning, because the patterns are realistic and the code
> runs fast. It also manufactures a precision the original survey never had, so
> the caveat travels with every number: these figures are for planning and
> practice, not population claims about Brazil.

**🏠 Homework depth: run this on your own.**
**Before you ask:** write your own one-sentence guess for where the mean lands on
the 1-to-7 scale, and whether most people actually cluster there.

> 💡 **Gemini Prompt:** "This cell plots the full distribution of a 1-to-7 trust
> item and marks its mean, median, and mode: [paste the next cell]. Explain why
> the mean can land in a spot where almost nobody actually answered, and what the
> shape tells a reader that the mean alone hides. Then argue against your own
> explanation the way a skeptical reader would."
>
> **After running, verify (counters *confident fabrication*: a fluent story about
> the shape can still misstate the numbers under it):**
> - [ ] Read the printed mean, median, and mode off your own output, and confirm
>   Gemini's story uses those three numbers, not ones it guessed.
> - [ ] Check its claim about where the mean falls against the actual bar heights
>   in your plot. If the shape it describes is not the shape you see, trust the plot.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

### 🛠️ Hands-On: read one distribution

Run the cell. It plots the full distribution of `trust_police` and marks the
mean. Read the shape before you read the next markdown cell.

In [ ]:
# The distribution of one trust item, with its mean marked.
tp = lapop["trust_police"]
counts = tp.value_counts().sort_index()

fig, ax = plt.subplots()
ax.bar(counts.index, counts.values, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Trust in the police (1 = none, 7 = a lot)")
ax.set_ylabel("Number of respondents")
ax.set_title("Distribution of reported trust in the police\n(lapop_brazil teaching resample, n = 10,000)")
ax.axvline(tp.mean(), color="#C44E52", linestyle="--", linewidth=2,
           label=f"mean = {tp.mean():.2f}")
ax.legend()
plt.tight_layout()
plt.show()

print("Counts per scale point:")
print(counts.to_string())
print(f"\nmean = {tp.mean():.2f}   median = {tp.median():.0f}   mode = {tp.mode().iloc[0]:.0f}")

In [ ]:
# Self-check: the mean sits between two peaks, not on one.
assert abs(tp.mean() - 4.43) < 0.02, "mean drifted — investigate"
assert tp.mode().iloc[0] == 7, "mode should be 7"
assert counts.loc[7] > counts.loc[4], "expected more 7s than 4s"
print("✓ Self-check passed: mean ≈ 4.43, median is 5, and the single most common answer "
      "is 7.\n  The mean lands in a dip between a low lump and a high pile.")

**Reading the evidence.** The mean is about 4.43, which sounds like a lukewarm
middle. Look where 4.43 falls, though: it sits in a dip. The single most common
answer is 7 (over 2,100 people), there is a real lump of 1s (about 1,280 people),
and the answers split rather than cluster. The mean of 4.43 describes almost no
actual respondent. Reporting it alone would suggest a country of moderates that
this distribution does not show. The lesson to carry: the average is a summary of
the distribution, never a substitute for it.

Now two variables at once. Do people who trust the police also tend to trust the
military? And can you fold several trust items into one **index**?

**🏠 Homework depth: run this on your own.**
**Before you ask:** write one sentence on what a positive police-and-military trust
correlation would, and would not, prove.

> 💡 **Gemini Prompt:** "This cell shows how average trust in the military rises
> across levels of trust in the police, then builds an index by averaging three
> institutional-trust items: [paste the next cell]. Explain what a positive
> relationship does and does not tell me, and one risk of averaging three items
> into one index. Then give me a competing explanation for why the two trust items
> move together that does NOT involve one causing the other."
>
> **After running, verify (counters *silent scope change*: a partner that quietly
> upgrades 'related' to 'causes' has answered a different question than you asked):**
> - [ ] Confirm the direction it describes matches your printed table: does average
>   military trust actually rise as police trust rises, and is the correlation the
>   value your cell reported?
> - [ ] Flag any sentence that treats the relationship as causal. This design never
>   intervened, so "because" is off the table. Rewrite it as "related."
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

### 🛠️ Hands-On: one relationship and one index

Run the cell. It reports how average trust in the military changes across levels
of trust in the police (a relationship), then builds a single institutional-trust
index from three items (police, military, supreme court).

In [ ]:
# A relationship: mean trust_military at each level of trust_police.
rel = lapop.groupby("trust_police")["trust_military"].mean().round(2)
corr = lapop["trust_police"].corr(lapop["trust_military"])
print("Average trust in the military at each level of trust in the police:")
print(rel.to_string())
print(f"\nCorrelation (trust_police, trust_military) = {corr:.2f}  (positive, moderate)")

# An index: average three institutional-trust items into one 1-7 score.
items = ["trust_police", "trust_military", "trust_supreme_court"]
lapop["institutional_trust"] = lapop[items].mean(axis=1)
idx = lapop["institutional_trust"]
print(f"\nInstitutional-trust index (mean of {', '.join(items)}):")
print(f"  mean = {idx.mean():.2f}   sd = {idx.std():.2f}   range = {idx.min():.1f} to {idx.max():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(rel.index, rel.values, marker="o", color="#4C72B0")
axes[0].set_xlabel("Trust in the police (1-7)")
axes[0].set_ylabel("Mean trust in the military")
axes[0].set_title("A relationship: the two trust items move together")
axes[1].hist(idx, bins=np.arange(1, 7.5, 0.5), color="#55A868", edgecolor="white")
axes[1].axvline(idx.mean(), color="#C44E52", linestyle="--", linewidth=2,
                label=f"index mean = {idx.mean():.2f}")
axes[1].set_xlabel("Institutional-trust index (1-7)")
axes[1].set_ylabel("Number of respondents")
axes[1].set_title("An index: three items combined into one score")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the relationship is positive and the index is a sensible 1-7 measure.
assert corr > 0.4, "expected a clearly positive relationship"
assert rel.loc[7] > rel.loc[1], "military trust should rise with police trust"
assert 1.0 <= idx.min() and idx.max() <= 7.0, "index must stay on the 1-7 scale"
print(f"✓ Self-check passed: military trust rises from {rel.loc[1]:.2f} (police=1) to "
      f"{rel.loc[7]:.2f} (police=7);\n  the index averages three items onto one honest 1-7 scale.")

**Reading the evidence.** The two trust items move together: people who trust
the police more report more trust in the military too, and the correlation is a
moderate positive 0.46. That is a **relationship**, a pattern in who answers how.
It is not a cause. This observational description never intervened, so nothing
here says trusting police *makes* anyone trust the military. Both could rise with
some third thing you never measured. The index does its own job: it smooths three
noisy single items into one steadier score, which is useful, but it also hides
disagreement. A person who trusts police 7 and the courts 1 lands at a middling 4,
the same as a lukewarm-on-everything respondent. An index is only as honest as
the items you put in it, and only worth reporting alongside what it flattens.

> **A question that often comes up here:** *"A positive correlation this clear
> feels like it must mean something causal. Why not?"* Because a relationship
> only tells you two things vary together in these data. It cannot tell you which
> one moved the other, or whether a third variable moved both. Turning "they move
> together" into "one causes the other" needs a different design entirely, the
> one that opens next week. Until then, an honest description says "related," not
> "because."

---

## 5. Nonresponse, Missingness, and the Generalization Boundary

**Guiding question:** *when does a description of your sample earn the right to
become a statement about a population, and when does it not?*

So far you have described the sample honestly. The last step is the riskiest one:
reaching past the sample to the population. Two words govern that step.

- **Nonresponse**: the people you drew into your sample but who never gave you an
  answer. Example: you email 500 people and 300 reply. The 200 silent ones are
  nonresponse, and they are not a random slice.
- **Missingness**: values that are absent from your data, whether a whole person
  or a single skipped question. The dangerous kind is missingness that relates to
  the very thing you are measuring. Example: people with the lowest trust in
  police are also the least willing to answer a trust-in-police survey.

Missingness is **rarely random**. When who is missing relates to what you
measure, the average of the people who *did* answer drifts away from the sample
truth, and any population claim built on it inherits that drift. Watch it happen.

### 🔮 Pause & Predict

The next cell keeps the same trust-in-police answers, but now lets *low-trust*
people drop out more often than high-trust people. Before you run it, commit: when
the low-trust answers go quiet, will the estimate you report drift *up* or *down*
from the true sample mean of about 4.43? Write one sentence naming the direction
and why. Do not run the cell until you have committed.

### YOUR ANSWER HERE:

**Up or down, and why:**

---

### 🛠️ Hands-On: the nonresponse stress test

Run the cell. It keeps the same trust-in-police answers, but now lets low-trust
people drop out more often, exactly the missingness that is not random. It then
compares the estimate you would report from respondents against the true sample
value you started with.

In [ ]:
# Nonresponse that is NOT random: lower trust means lower chance of answering.
# Re-seed a fresh generator so this result is reproducible on its own (seed 464).
rng_nr = np.random.default_rng(SEED)

true_sample_value = tp.mean()                      # the honest sample average
p_respond = np.clip(0.22 + 0.085 * (tp - 1), 0, 1) # trust 1 -> .22, trust 7 -> .73
responded = rng_nr.random(len(lapop)) < p_respond
observed_value = tp[responded].mean()              # what you'd report from respondents
response_rate = responded.mean()

print(f"Response rate: {response_rate:.1%}  (so nonresponse is {1 - response_rate:.1%})")
print(f"True sample mean trust_police      : {true_sample_value:.3f}")
print(f"Estimate from respondents only     : {observed_value:.3f}")
print(f"Nonresponse moved the estimate by  : {observed_value - true_sample_value:+.3f} points")

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(["True sample\nvalue", "Respondents-only\nestimate"],
       [true_sample_value, observed_value],
       color=["#4C72B0", "#C44E52"], edgecolor="white")
ax.set_ylim(1, 7)
ax.axhline(4, color="gray", linestyle=":", linewidth=1.2, label="scale midpoint (4)")
ax.set_ylabel("Mean trust in the police (1-7)")
ax.set_title("Nonresponse that is not random pushes the estimate off the sample truth")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: dropping low-trust people pushes the reported estimate UP.
assert 0.45 < response_rate < 0.56, "response rate drifted — is the seed 464?"
assert observed_value - true_sample_value > 0.5, "expected a clear upward drift"
print(f"✓ Self-check passed: with about {1 - response_rate:.0%} nonresponse, the "
      f"respondents-only estimate ({observed_value:.2f})\n  overstates the true sample "
      f"value ({true_sample_value:.2f}) by more than half a point. Same data, biased picture.")

### 🔍 Reading the Evidence: where the description must stop

The people who answered were not a fair slice of the sample: the low-trust
answers went quiet, so the reported estimate drifted up by more than half a
point. In the cell below, write three things. First: the honest headline
sentence you could report from the respondents, worded to stop exactly where your
coverage and response rate stop. Second: the tempting sentence you must NOT
write, the one that treats the respondents' average as the population's average.
Third: name the crossing that a population claim would need to pay for, and one
piece of evidence that would license it.

### YOUR ANSWER HERE:

**My honest headline sentence (stops at coverage + response):**

**The sentence I must NOT write:**

**The crossing a population claim must pay for, and what would license it:**

---

**🔴 Live in class: we run this one together.**
You just wrote an honest headline sentence in the cell above. Do not let Gemini
write your caveat for you. You wrote it. Now make Gemini try to break it.

> 💡 **Gemini Prompt:** "Here is a boundary sentence I wrote about a survey
> estimate: '[paste your honest headline from the cell above].' The estimate comes
> from respondents only, and lower-trust people answered less often, so the
> reported mean of about 5.07 sits above the true sample mean of 4.43. Act as a
> hostile peer reviewer. Name every place my sentence over-reaches, under-states
> the bias, or hides the direction of the drift. Do not rewrite it for me. List the
> specific weaknesses so I can fix them myself."
>
> **After running, verify (counters *sycophantic agreement*: an AI that praises the
> caveat you wrote has reviewed your ego, not your evidence):**
> - [ ] If every objection is mild, or it calls your sentence "well-balanced," push
>   back: "assume it is wrong, name the single worst flaw." Weak praise is a red flag.
> - [ ] Check each objection against your printout: the drift is about +0.65
>   *upward*, so reject any objection claiming the estimate runs low.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

---

### ⏸ In class through here

**You have reached the end of Lecture 2's in-class block.** Everything below
deepens the same ideas and feeds your M4 audit. Finish it before Friday's studio:
adapt the attack prompt to your own project, interrogate what comes back, make the
AI-free reach decision, repair the three claims, and draft your project's audit
spine. The 🧑‍⚖️ Human-Only Checkpoint is the never-delegate core of M4, so give it
real time.

---

### 🔁 Modify the Prompt

Above, you had Gemini attack a caveat *you* wrote for the teaching estimate. Now do
the same for your own project. First, in one or two sentences, write the honest
boundary for your project's estimate yourself. Do not let AI draft it. Then adapt
the attack prompt so Gemini red-teams *your* sentence, not the survey's.

> **Base prompt (the attack, from above):** "Here is a boundary sentence I wrote:
> '[your sentence].' Act as a hostile peer reviewer and name every place it
> over-reaches, under-states the bias, or hides the direction of the drift. Do not
> rewrite it for me."

Rewrite it so it names *your* frame, *your* outcome, and the group *you* most worry
is missing not-at-random. In the cell below: paste the boundary sentence you wrote,
paste your adapted attack prompt, predict the single strongest objection Gemini will
raise, then run it and note whether your prediction held.

### YOUR ANSWER HERE:

**The boundary sentence I wrote for my project (AI did not draft this):**

**My adapted attack prompt (my frame, my outcome, my missing group):**

**The strongest objection I predict Gemini will raise:**

**What Gemini actually raised, and how I fixed my sentence:**

---

### 🔬 Interrogate the Output

Gemini just handed you a list of objections to the caveat *you* wrote. Do not
accept the attack as automatically right either. Interrogate the attack itself
against the numbers, using four checks. Answer each in the cell below.

- **Claims:** does each objection point to a real flaw, and does it match your
  output? Your respondents-only estimate came out too *high* (about 5.07 vs 4.43),
  so an objection that you under-stated an *upward* bias is fair, but one claiming
  the estimate runs low is simply wrong.
- **Assumptions:** what does the attack assume about *why* people are missing? Is
  that assumption yours to make, or did the tool invent a mechanism you never named?
- **Missing information:** which real weakness did the attack *miss*? A hostile
  reviewer that skips your response rate or the size of the drift left the most
  important objection on the table.
- **Overstatements:** does any objection sound more certain than the evidence
  supports? Flag the exact words.

And one rule that governs the whole notebook: **code running without errors is not
the same as code being correct.** A cell can execute cleanly and still compute the
wrong quantity. Verify the *number*, not just the green check.

### YOUR ANSWER HERE:

**Claims (does each objection match my numbers?):**

**Assumptions (did the attack invent a mechanism?):**

**Missing information (the real weakness the attack skipped):**

**Overstatements (exact words):**

---

### 🧑‍⚖️ Human-Only Checkpoint

Close Gemini for this one. No AI, no notebook search. This is a never-delegate
decision: the reach of your own project's evidence is yours to declare and
defend. In the cell below, write, in your own words:

1. Your project's **population / accessible population / frame / sample** stack,
   one line each.
2. **Two sentences, the exact pair your M4 boundary grades.** First, the licensed
   population sentence your frame and expected response actually support, worded to
   stop exactly where your evidence stops, with its uncertainty. Second, the named
   **silent upgrade** you are forbidding: the tempting sentence that would quietly
   swap your frame for the whole population, written out so you can refuse it on sight.

If you cannot write the licensed sentence honestly yet, that is a finding, not a
failure. It tells you which crossing you still have to pay for.

### YOUR ANSWER HERE:

**Population:**

**Accessible population:**

**Frame:**

**Sample:**

**Sentence 1, the population claim my frame licenses (with its uncertainty):**

**Sentence 2, the silent upgrade I forbid:**

---

### 📝 Practice: repair three reach claims

*(Human-first: repair all three yourself before any AI. Catching the silent
upgrade by eye is the graded skill.)*

Each claim below crosses from a sample to a population without paying for it.
Rewrite each so it either stays honestly inside the sample, or names the exact
coverage limit a population version would have to disclose. Answer in the cell
that follows.

- **A.** "We surveyed the people who filled out our club's form, so 71% of
  undergraduates prefer online exams."
- **B.** "Our air sensors sit in the newest three buildings and read clean, so
  campus air quality is fine."
- **C.** "Sixty percent of respondents approve, and 40% did not answer, so
  approval in the population is 60%."

### YOUR ANSWER HERE:

**A repaired:**

**B repaired:**

**C repaired:**

---

### 🎯 Project Transfer: the spine of your M4 audit

*(Human-first: draft every piece yourself. AI may critique it after, but the audit
you defend at the studio has to be reasoned by you.)*

This is where the week comes together into your **M4 observational descriptive
design audit**. In the cell below, draft the audit's spine for *your* project:

1. **The stack:** your target population, accessible population, frame, and
   sample, one line each (bring it forward from the Human-Only Checkpoint).
2. **The selection DAG:** in words or a quick sketch, the arrows from target to
   observed, and the one **selection influence** you most fear bends both who you
   reach and what you measure.
3. **The measurement ladder and coverage note:** your **concept → construct →
   indicator** for the key measure, and the one variable you most worry is missing
   not-at-random (name it coverage, nonresponse, or missingness).
4. **The boundary line (two sentences):** first, the population claim your frame
   licenses with its uncertainty; second, the named silent upgrade your design
   forbids. These are the exact two sentences M4 grades.

These four pieces are what you will walk a skeptic through at Friday's studio.
Write them so a reader who has never seen your project could locate the exact edge
of your evidence.

### YOUR ANSWER HERE:

**1. The stack (target / accessible / frame / sample):**

**2. The selection DAG (arrows + the selection influence I most fear):**

**3. Measurement ladder and coverage note (concept, construct, indicator, the missing-not-at-random risk):**

**4. My boundary line, sentence 1 (licensed claim) and sentence 2 (forbidden silent upgrade):**

---

### 📒 AI Research Ledger

Log every AI use from this notebook in the ledger. One worked row is filled in as
a model, and it models the discipline this week teaches: **you drafted the caveat,
the AI attacked it.** This notebook offers six prompts, one live per lecture and
four homework-depth, so your ledger carries a row for each one you actually ran,
not a fixed count. The ledger is a graded habit, not paperwork: it is how you show
your work was verified.

| Task delegated | Tool used | Prompt | Output summary | Decision | Verification method | Remaining concern | Responsible researcher |
|---|---|---|---|---|---|---|---|
| Red-team the boundary sentence I wrote for the nonresponse estimate | Gemini | "Here is a caveat I wrote: [sentence]. Act as a hostile reviewer and name every over-reach; do not rewrite it." | Three objections to my sentence: one fair (I under-stated how far the estimate runs high), two misread the direction | Accepted the fair objection and tightened my sentence; rejected the two that claimed the estimate runs low | Recomputed the drift myself (+0.65 upward); kept only the objection matching that direction | Real surveys will not hand me the true value to check an attack against | *(your name)* |
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |

---

### 🛡️ Exit Defense

Defense #04 — write, in your own words:

1. **The claim I can defend:** one bounded sentence, drawn from today's data or
   your own project, that you would put your name on.
2. **Its boundary:** what this evidence does NOT establish. Name the compass
   position (description or generalization) and whether you licensed the
   sample → population crossing or left it unpaid.
3. **My uncertainty and limitations:** one sentence naming your coverage gap,
   nonresponse risk, or the resample caveat.
4. **AI check:** what you delegated to Gemini, whether you **accepted, changed, or
   rejected** what it returned, and the specific check that decided it (a number
   you recomputed, a source you retrieved, a direction you confirmed).

### YOUR ANSWER HERE:

**1. The claim I can defend:**

**2. Its boundary (compass position + crossing licensed or not):**

**3. My uncertainty and limitations:**

**4. AI check (what I delegated, how I verified):**

---

## 6. Wrap-Up

Across two lectures you built an observational descriptive design from the ground
up. You separated the target population, the accessible population, the frame, and
the sample, and drew the selection DAG that shows where people drop out. You
watched random draws huddle around a frame's truth while a convenience channel
missed it by nearly twelve years, biased not by luck but by *who it could reach*.
You summarized a real survey with a distribution, a relationship, and an index,
and read what each reveals and hides. Then you stress-tested nonresponse and saw a
clean estimate drift off the truth, which set the exact boundary past which a
description may not speak for a population.

> **"A description earns the word 'population' only when a sampling strategy and
> an honest response rate pay for the crossing. A claim that grows from 'in my
> sample' to 'in the population' with nothing paying for it is the silent
> upgrade, and it is the one move this week exists to catch."**

Next week the causal question opens. Once description and generalization are on
the table, the natural next ask is *why*, and answering it honestly takes a
different design. Your M4 audit presents at Friday's studio, where M5 kicks off
with the causal identification strategy. Bring your population/frame/sample stack
and your boundary line. They are about to meet the word "because."

---

## 7. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 15 'Observational: descriptive' | population/frame/sample stack, measurement, nonresponse, distributions/relationships/indices, the generalization boundary | adapted (concept-level, honors non-quant audience)*
- *DeclareDesign observational-descriptive design library | the observational descriptive pathway framing (Model + Inquiry + Data strategy) | referenced*
- *la_voter_file.csv | rdss package data | party/age used as a known frame for seeded sampling and selection demonstrations | adapted (the file stands in as the 'known world')*
- *lapop_brazil.csv | rdss package data | trust items summarized as a distribution, a relationship, and an index, then a seeded nonresponse stress-test | adapted (10,000-row resample-with-replacement; caveat taught explicitly)*
- *fresh | the selection DAG + the coverage-and-nonresponse note + the boundary line | newly-constructed-from-source-concept*
- *callingbullshit.org (public index) | selection-bias material | linked as optional study, index pages only | referenced*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 15 'Observational: descriptive'. Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).
- *(Optional)* Bergstrom, C. & West, J. *Calling Bullshit*, selection-bias case
  index: [callingbullshit.org/tools](https://callingbullshit.org/tools/).

---

<center>

Thank you!

</center>